# Loop calling and evaluation tutorial

This public tutorial includes explanations, command examples and embedded figures. Commands use the companion analysis-code checkout as their working directory. Scientific inputs and native software environments are configured separately.

This tutorial follows the analysis from input contact maps to loop calls,
task-specific metrics and workflow summaries. The commands and Python examples
document the companion analysis implementation. They are displayed as code
blocks: the notebook can be read without a running kernel or access to the data.

The Lee2019 ODC example compares **Mustache + Raw** and **Mustache + iEnhance**.
The other caller commands and the full-benchmark summary steps are shown in
their own sections.

```text
Matched Raw / imputed inputs
          |
     Loop calling --------------------> Loop num and caller runtime
          |
          +--> Mapped-reference Dice --> Cell-number trajectories
          +--> APA on the Raw map
          +--> RNA correlation
          +--> mCG correlation
                         |
              Per-metric tables
                         |
           Rank-product changes / weighted workflow ranking
```


Use the [shared path setup](tutorials.ipynb#Shared-path-prefix) for the commands below:

```bash
source tutorials/paths.sh
cd "${BENCHMARK_ROOT}/code"
```

```python
from pathlib import Path
import os

BENCHMARK_ROOT = Path(os.environ["BENCHMARK_ROOT"])
```


## 1. Prepare the inputs

Bulk callers use cell-type pseudo-bulk contact maps at **10 kb** resolution.
The Raw and imputed query maps represent the same cells at each cell-number
condition. scHiCluster uses its native single-cell loop-calling branch, described
in section 5.

The Lee2019 ODC inputs are:

| Input | Contents | Used for |
|---|---|---|
| `ODC.raw.100.cool` | Raw contacts from 100 ODC cells | Raw query calling |
| `ODC.impute.100.iEnhance_10k.cool` | The matched iEnhance contact map | Imputed query calling |
| `ODC.raw.all.cool` | Raw contacts from all ODC cells | Common APA map |
| `Tian2023_ODC_raw_all_Mustache.tsv` | Prespecified mapped-reference loops | Dice comparison |
| MouseBrain loop files, Raw maps, RNA and DEG tables | The same 17 matched cell types | RNA correlation |
| Lee2019 loop files, Raw maps, cell lists and MCDS | The same 14 matched cell types | mCG correlation |

For a cell-number series, prepare the matched Raw and imputed maps for each
condition before loop calling. Contact preparation and imputation are documented
in [data_preparation](data_preparation.ipynb). Detailed file formats and
the existing local input locations are in DATA_REQUIREMENTS.md ([call_loop/DATA_REQUIREMENTS.md](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/DATA_REQUIREMENTS.md) in this folder).

Metadata read from `D:/data_new/lee2019/lee2019/ODC_raw_10k_line/ODC.raw.100.cool`:

```json
{
  "bin-size": 10000,
  "nbins": 287509,
  "nchroms": 22,
  "nnz": 18648440
}
```

The Raw Mustache loop file begins with:

Source: `F:/project/test1/final_code2/call_loop/result/raw_mustache_example/ODC.raw.100.10k_mustache_loop.tsv` (first 3 lines).

```text
BIN1_CHR	BIN1_START	BIN1_END	BIN2_CHROMOSOME	BIN2_START	BIN2_END	FDR	DETECTION_SCALE
chr1	30790000	30800000	chr1	30900000	30910000	0.04316786880353285	2.111212657236631
chr1	43230000	43240000	chr1	43370000	43380000	0.038127131296458706	2.111212657236631
```


## 2. Set the paths

The following Python examples share these path variables. Replace `/path/to/`
with the locations of the repository, inputs and desired outputs when using the
code outside this tutorial.

```python
from pathlib import Path
import sys
import os

REPO = (BENCHMARK_ROOT / "code")
TOOL = REPO / "call_loop/tool"
OUTPUT = Path(os.environ["LOOP_OUTPUT_DIR"])

for folder in (TOOL, TOOL / "scripts", TOOL / "callers", TOOL / "plots"):
    sys.path.insert(0, str(folder))

RAW_100 = (BENCHMARK_ROOT / "data/lee2019/ODC_raw_10k_line/ODC.raw.100.cool")
IMPUTED_100 = (BENCHMARK_ROOT / "data/lee2019/bulk_impute_10k/ODC/ODC.impute.100.iEnhance_10k.cool")
RAW_ALL = (BENCHMARK_ROOT / "data/lee2019/ODC_raw_10k_line/ODC.raw.all.cool")
REFERENCE = (BENCHMARK_ROOT / "records/data/raw_dice_example/Tian2023_ODC_raw_all_Mustache.tsv")
```

Calling and evaluation use their respective software environments; the caller
commands below are not intended to share one environment. Installation and
version information are in [software notes](#Software-and-implementation).


## 3. Call the matched Raw and imputed maps with Mustache

The example runs Mustache with a 10-kb resolution, a 5-Mb maximum distance and
`-pt 0.05`. The native wrapper records the actual command, input and output
hashes, loop count, software environment and caller wall-clock time.

```python
from run_raw_mustache_example import run_example as call_mustache

NATIVE_PYTHON = (BENCHMARK_ROOT / "software/Mustache/Scripts/python.exe")

for method, cool_file, folder in [
    ("Raw", RAW_100, "raw"),
    ("iEnhance", IMPUTED_100, "ienhance"),
]:
    call_mustache(
        cool_file=cool_file,
        native_python=NATIVE_PYTHON,
        output_dir=OUTPUT / "calling" / folder,
        input_method=method,
        processes=4,
    )
```

Each run produces a native loop TSV, `loop_num_runtime.csv`, `mustache.log`
and `run_manifest.json`. In this ODC/100-cell example, the query filenames are:

```python
QUERIES = {
    "Raw": OUTPUT / "calling/raw/ODC.raw.100.10k_mustache_loop.tsv",
    "iEnhance": OUTPUT / "calling/ienhance/ODC.impute.100.iEnhance.10k_mustache_loop.tsv",
}
```

Implementation: run_raw_mustache_example.py ([call_loop/tool/callers/run_raw_mustache_example.py](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/tool/callers/run_raw_mustache_example.py) in this folder).


## 4. Run the other bulk callers

The following command examples preserve the settings in
callers.py ([call_loop/tool/callers/callers.py](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/tool/callers/callers.py) in this folder). Each caller receives the selected
Raw or imputed 10-kb pseudo-bulk map. File paths below are placeholders for that
caller's inputs, checkpoint and output directory.

### FitHiC2

Convert the Cooler pixels to interaction and fragment tables, run FitHiC2 and
retain records with a Q value below 0.01.

```bash
cooler balance "${BENCHMARK_ROOT}/data/lee2019/ODC_raw_10k_line/ODC.raw.100.cool"

python call_loop/tool/callers/fithic2/convert_rawbulk2fithic.py \
    --cool "${BENCHMARK_ROOT}/data/lee2019/ODC_raw_10k_line/ODC.raw.100.cool" \
    --iter "${LOOP_OUTPUT_DIR}/query.interaction" \
    --frag "${LOOP_OUTPUT_DIR}/query.fragment"

gzip "${LOOP_OUTPUT_DIR}/query.interaction"
gzip "${LOOP_OUTPUT_DIR}/query.fragment"

fithic \
    -i "${LOOP_OUTPUT_DIR}/query.interaction.gz" \
    -f "${LOOP_OUTPUT_DIR}/query.fragment.gz" \
    -o "${LOOP_OUTPUT_DIR}/fithic_spline" \
    -r 10000

gunzip "${LOOP_OUTPUT_DIR}/fithic_spline/FitHiC.spline_pass1.res10000.significances.txt.gz"

python call_loop/tool/callers/fithic2/2tsv.py \
    --fn "${LOOP_OUTPUT_DIR}/fithic_spline/FitHiC.spline_pass1.res10000.significances.txt" \
    --of "${LOOP_OUTPUT_DIR}/query_10k_fithic2_loop.tsv" \
    --qv 0.01
```

The conversion scripts retain the caller-specific significance fields and
write the 10-kb anchor coordinates used by the downstream readers.

### HICCUPS / HiCPeaks

The human-genome example below uses chromosomes 1–22. For mouse inputs,
use the corresponding autosomal chromosome list.

```bash
pyHICCUPS \
    -p "${BENCHMARK_ROOT}/data/lee2019/ODC_raw_10k_line/ODC.raw.100.cool" \
    -O "${LOOP_OUTPUT_DIR}/query.10k_hiccups_loop.bedpe" \
    -C 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 \
    --pw 1 2 4 \
    --ww 3 5 7 \
    --maxapart 5000000 \
    --only-anchors
```

### Peakachu

First inspect `peakachu depth` to select the suggested official model. Replace
`{depth}` in the model filename with that depth category, then score and pool
the candidate loops.

```bash
peakachu depth -p "${BENCHMARK_ROOT}/data/lee2019/ODC_raw_10k_line/ODC.raw.100.cool"

peakachu score_genome \
    -r 10000 \
    -p "${BENCHMARK_ROOT}/data/lee2019/ODC_raw_10k_line/ODC.raw.100.cool" \
    -u 500 \
    -O "${LOOP_OUTPUT_DIR}/query.10k_peakachu_loop.bedpe" \
    -m ${PEAKACHU_MODEL}

peakachu pool \
    -r 10000 \
    -i "${LOOP_OUTPUT_DIR}/query.10k_peakachu_loop.bedpe" \
    -o "${LOOP_OUTPUT_DIR}/query.10k_peakachu_pooled_loop.bedpe" \
    -t 0.95
```

### HiCFoundation

Download the checkpoints from the official
[Hugging Face model repository](https://huggingface.co/wang3702/hicfoundation_models/tree/main),
as listed in the [HiCFoundation installation instructions](https://github.com/Noble-Lab/HiCFoundation#4-download-the-trained-hicfoundation-model).
Store them in the `hicfoundation_model/` directory under the HiCFoundation installation.

On this computer, the installation is `D:/hicfoundation/HiCFoundation/`;
the model directory is `D:/hicfoundation/HiCFoundation/hicfoundation_model/`.
The corresponding WSL download commands are:

```bash
cd "${BENCHMARK_ROOT}/software/HiCFoundation"
mkdir -p hicfoundation_model
cd hicfoundation_model
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_pretrain.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_reproducibility.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_loop.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_loop_lc.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_resolution.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_epigenomic.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_schic.pth.tar
cd ..
```

| Workflow | Task | Checkpoint relative to the installation |
|---|---|---|
| Pseudo-bulk imputation | 3 | `hicfoundation_model/hicfoundation_resolution.pth.tar` |
| Loop calling in this workflow | 2 | `hicfoundation_model/hicfoundation_loop.pth.tar` |

For task-2 loop calling, select the loop checkpoint:

```bash
HICFOUNDATION_ROOT="${BENCHMARK_ROOT}/software/HiCFoundation"
HICFOUNDATION_SCRIPT="${HICFOUNDATION_ROOT}/inference.py"
HICFOUNDATION_MODEL="${HICFOUNDATION_ROOT}/hicfoundation_model/hicfoundation_loop.pth.tar"
RAW_100="${BENCHMARK_ROOT}/data/lee2019/ODC_raw_10k_line/ODC.raw.100.cool"

cooler balance "$RAW_100"
python "$HICFOUNDATION_SCRIPT" \
    --input "$RAW_100" \
    --batch_size 4 \
    --resolution 10000 \
    --task 2 \
    --input_row_size 224 \
    --input_col_size 224 \
    --stride 100 \
    --bound 500 \
    --model_path "$HICFOUNDATION_MODEL" \
    --output "$HICFOUNDATION_LOOP_OUTPUT_DIR"
```

The workflow uses the generated `HiCFoundation_loop_0.5.bedpe` file.


## 5. Prepare the native scHiCluster loop workflow

scHiCluster uses per-cell imputed Coolers and cell-type metadata. Its
preparation script groups cells, writes the per-group cell table and
`Snakefile_master`, then creates `snakemake_cmds.txt`.

For Lee2019, use hg38 resources and a cell table pointing to the 10-kb
single-cell Coolers. In
[prepare_loop.py](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/tool/callers/schicluster/prepare_loop.py),
replace the dataset-settings block with the Lee2019 paths and metadata columns:

```python
from pathlib import Path
import json
import os
import pandas as pd

metadata = pd.read_csv(os.environ["SCHICLUSTER_LOOP_METADATA"], sep="\t", index_col="cell_id")
cell_table = pd.read_csv(
    os.environ["SCHICLUSTER_LOOP_CELL_TABLE"],
    sep="\t", header=None, names=["cell_id", "cool_path"], index_col="cell_id",
)
cell_table["cluster"] = (
    metadata.loc[cell_table.index, "cell_type"].astype(str)
    .str.replace(r"[^A-Za-z0-9_-]+", "_", regex=True).str.strip("_")
)
outdir = str(Path(os.environ["SCHICLUSTER_LOOP_OUTPUT"])) + "/"
loop_dir = outdir
params = {
    "cpu": 96,
    "resolution": 10000,
    "chrom_size_path": json.dumps(str(BENCHMARK_ROOT / "code/resources/genome/hg38.autosomal.chrom.sizes")),
    "black_list_path": json.dumps(os.environ["HG38_BLACKLIST"]),
}
```

The cell table defines the input membership. Metadata keeps the original
cell-type labels, while directory names use the cleaned `cluster` values.
Run the preparation script from its directory:

```bash
cd call_loop/tool/callers/schicluster
python prepare_loop.py
```

Then launch the generated workflows:

```bash
bash "${SCHICLUSTER_LOOP_OUTPUT}/snakemake_cmds.txt"
```

The native scHiCluster analysis produces complete-loop and loop-summit outputs.
Use the complete-loop output for the standard metric workflow; the supplied
APA code also supports the output-definition comparisons shown later.



## 6. Record loop num and caller runtime

Loop num is counted from the native caller output. Caller-specific readers
then convert those outputs to the common chromosome and anchor-coordinate
representation used for metric calculation.

For the Mustache example, `loop_num_runtime.csv` stores the native count and
recorded wall-clock time. Timing starts when the caller subprocess is launched
and ends when it completes, including native imports, map reading, normalization
and output writing. Environment setup, preflight and file hashing occur outside
that interval.

```python
import pandas as pd

raw_calling = pd.read_csv(OUTPUT / "calling/raw/loop_num_runtime.csv")
raw_calling[["loop_num", "runtime_seconds"]]
```


## 7. Compare loops with the mapped reference

Both query loop sets use the same prespecified Tian2023 ODC all-cell Raw
Mustache comparator. The readers retain cis interactions on the 10-kb grid,
order the anchors and remove duplicate coordinate pairs before matching.

```python
from run_raw_dice_example import run_example as evaluate_dice

for method, query in QUERIES.items():
    dice_summary, matched_pairs, dice_record = evaluate_dice(
        query_file=query,
        reference_file=REFERENCE,
        output_dir=OUTPUT / "dice" / method,
        input_method=method,
    )
```

For query set Q and mapped-reference set R, the concordance is:

```text
Dice = 2 × number of shared coordinate pairs / (number of query pairs + number of reference pairs)
```

The output contains the query count, reference count, intersection count and
Dice value; `matched_loop_pairs.tsv` records the shared pairs. The common
comparison functions are in tools.py ([call_loop/tool/tools.py](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/tool/tools.py) in this folder).


## 8. Repeat the comparison across cell numbers

Build matched Raw and imputed conditions in 100-cell increments and include the
all-cell endpoint. For example, the following derives the conditions from a
headerless cell table containing one selected cell per row:

```python
import pandas as pd

cell_table = pd.read_csv((BENCHMARK_ROOT / "data/lee2019/ODC_raw_10k_line/cell_table_all.txt"), sep="\t", header=None)
all_cell_num = len(cell_table)
cell_nums = list(range(100, all_cell_num + 1, 100))
if all_cell_num >= 100 and cell_nums[-1] != all_cell_num:
    cell_nums.append(all_cell_num)
```

Run the selected caller on each prepared contact map, then evaluate each loop
set against the same mapped reference. The repository's series interface is:

```python
from tools import run_dice_series

dice_trajectory = run_dice_series(
    query_dataset="lee2019",
    query_cell_type="ODC",
    input_method="raw",
    loop_tool="mustache",
    cell_nums=cell_nums,
    raw_root=INPUT / "raw_loop",
    imputed_root=INPUT / "imputed_loop",
    schicluster_root=INPUT / "schicluster_loop",
)
```

The roots follow the dataset/cell-type layout used by `build_loop_paths` in
tools.py ([call_loop/tool/tools.py](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/tool/tools.py) in this folder). Repeat for the matched imputed input. The native
scHiCluster series uses `input_method="schicluster"`. Plot loop num alongside
Dice to show how output quantity and loop-set agreement change with cell number.


## 9. Calculate APA on the common Raw contact map

Evaluate both native query loop sets on the Lee2019 ODC all-cell Raw map.
The example uses 10-kb bins, a 10-bin window on each side, cis separations from
100 kb to 5 Mb and unbalanced contact counts.

```python
from run_raw_apa_example import run_example as evaluate_apa

for method, query in QUERIES.items():
    apa_summary, apa_matrix, apa_record = evaluate_apa(
        cool_file=RAW_ALL,
        loop_file=query,
        output_dir=OUTPUT / "apa" / method,
        collection_dir=OUTPUT / "tables/apa",
        input_method=method,
    )
```

The aggregate matrix is 21 × 21. The implemented P2LL calculation is:

```python
center = apa_matrix[9:12, 9:12].mean()
background = apa_matrix[0:3, 0:3].mean()
p2ll = center / background if background > 0 else 0.0
```

Outputs include the eligible and valid loop counts, P2LL, the saved matrix and
PDF/PNG heatmaps.

For other callers, run_apa ([call_loop/tool/tools.py](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/tool/tools.py) in this folder) accepts `loop_tool`, `loop_variant`
and `analysis_scope`. The explicit `fithic2_filtered_clustered` and
`schicluster_summit` variants support the Fig. 6e / Supplementary Fig. 10
output-definition comparisons in the existing code.


## 10. Evaluate RNA correlation

The MouseBrain example uses 17 matched cell types. It starts from per-type loop
sets, observed Raw maps, processed expression and 136 pairwise DEG files.
The two workflows use the same observed contact signal and RNA inputs.

The implementation selects genes from the supplied DEG comparisons, scales the
observed contact signal by `1600 / cell_num`, selects the top 1,000 eligible
loop–gene pairs by interaction variability and calculates their Pearson
correlations across cell types. The mean is exported separately for each workflow.

```python
from run_matched_raw_imputed_rna_example import run_example as compare_rna

rna_comparison = compare_rna(
    raw_loop_root=BENCHMARK_ROOT / "data/mousebrain/loop/others/raw",
    imputed_loop_root=BENCHMARK_ROOT / "data/mousebrain/loop/others/impute_renamed",
    cool_root=BENCHMARK_ROOT / "data/mousebrain/merged_data",
    rna_dir=BENCHMARK_ROOT / "data/rna/rna",
    deg_dir=BENCHMARK_ROOT / "data/rna/DEG/mousebrain/DEG",
    gene_bed=REPO / "resources/annotation/gencode.vM22.gene.slop2k.ID.bed",
    raw_output_dir=OUTPUT / "rna/raw",
    output_dir=OUTPUT / "rna/matched",
    loop_interval_end="end2",
    recompute_raw=True,
)
```

The worked example uses the supplied gene-interval annotation and the recorded
`end2` interval convention. The output includes selected loop–gene pairs,
per-pair correlations, interaction/RNA heatmaps and `rna_comparison.csv`.
The existing get_result.py ([call_loop/tool/get_result.py](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/tool/get_result.py) in this folder) also documents upstream
RNA preprocessing and configurable metric blocks for other callers.


## 11. Evaluate mCG correlation

The Lee2019 example uses 14 matched cell types. Methylated counts and coverage
are read from the MCDS at 5 kb and summed onto the 10-kb loop grid. The
implementation calculates the posterior mCG fraction, averages the values at
the two anchors and compares them with the observed Raw interaction signal
across cell types. It reports the mean Pearson correlation for the top 2,000
selected variable loops in each workflow.

```python
from run_matched_raw_imputed_mcg_example import run_example as compare_mcg

mcg_comparison = compare_mcg(
    mcds_dir=BENCHMARK_ROOT / "data/mcg",
    cool_root=BENCHMARK_ROOT / "data/lee2019/all_raw_merged",
    raw_loop_root=BENCHMARK_ROOT / "data/lee2019_raw_loops/type_loop",
    imputed_loop_root=BENCHMARK_ROOT / "data/lee2019_imputed_loops/type_changed_loop",
    raw_output_dir=OUTPUT / "mcg/raw",
    output_dir=OUTPUT / "mcg/matched",
    recompute_raw=True,
)
```

On Windows, this wrapper dispatches calculation to the configured Linux/WSL
ALLCools environment. Linux-side paths and the interpreter are described in
[software notes](#Software-and-implementation). The example retains the original signs of the
mCG correlations in its tables and paired heatmaps.


## 12. Summarize the metrics and compare workflows

First retain loop num, Dice, APA, RNA and mCG as separate result tables.
The full-benchmark summaries use the complete prespecified caller–input set
and selected dataset conditions, beyond the two-workflow worked example.

### Fig. 6c: Raw–imputed rank-product changes

The four components are loop num, mapped-reference Dice, mCG correlation and
RNA correlation. Loop num, Dice and RNA are ranked in descending order; the
mCG coefficient is sign-inverted before ranking. Ties receive the minimum rank.

```python
from tools import run_rank_product_shift
from result_tables import WORKFLOWS

rank_product = run_rank_product_shift(
    count_dice_path=BENCHMARK_ROOT / "records/data/loop_minmax/merged_count_f1_ranked.csv",
    mcg_path=BENCHMARK_ROOT / "records/data/loop_minmax/lee2019_cg_impute_diffloop.csv",
    rna_path=BENCHMARK_ROOT / "records/data/loop_minmax/mousebrain_rna_impute_diffloop.csv",
    expected_workflows=WORKFLOWS,
)
raw_imputed_changes = rank_product["rank_product_shift"]
```

For each workflow, multiply its four component ranks. For each caller, compare
the imputed rank product with its matched Raw rank product using the signed
logarithmic difference implemented in `run_rank_product_shift`.

### Fig. 5g, Fig. 6d and Supplementary Fig. 7: weighted ranking

Transform loop num as `log10(N + 1)`. Min–max normalize each component across
workflows within a dataset, average dataset-level normalized values within each
component and combine them with the following fixed weights:

| Component | Weight | Direction before normalization |
|---|---:|---|
| Loop num | 0.25 | Larger `log10(N + 1)` |
| Mapped-reference Dice | 0.15 | Larger |
| APA P2LL | 0.25 | Larger |
| mCG correlation | 0.20 | Sign-inverted coefficient |
| RNA correlation | 0.15 | Larger |

```python
from plot_loop_minmax import run_figures

metric_paths = {
    "count_dice": BENCHMARK_ROOT / "records/data/loop_minmax/merged_count_f1_ranked.csv",
    "apa": BENCHMARK_ROOT / "records/data/loop_minmax/APA_impute_all_cell_lines.csv",
    "mcg": BENCHMARK_ROOT / "records/data/loop_minmax/lee2019_cg_impute_diffloop.csv",
    "rna": BENCHMARK_ROOT / "records/data/loop_minmax/mousebrain_rna_impute_diffloop.csv",
}

scores, figure_files = run_figures(
    paths=metric_paths,
    output_dir=OUTPUT / "minmax",
    panel="both",
    top_n=10,
    show=False,
)
```

Fig. 6d displays the leading ten of 25 workflows and Supplementary Fig. 7
displays the full set with the same ordering. Fig. 5g uses the separate
native-workflow entry ([call_loop/tool/plots/plot_loop_native_minmax.py](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/tool/plots/plot_loop_native_minmax.py) in this folder), selecting the
six native workflows before normalization. See the
figure guide ([Loop figure tutorial](loop_figures.ipynb)) for the exact input tables and exports.


## 13. Read the outputs

Each stage writes its own outputs. The following layout summarizes the examples
above; it is an output guide rather than a directory that is loaded while
reading this notebook.

```text
outputs/
  calling/
    raw/                 native loops, loop_num_runtime.csv, run_manifest.json
    ienhance/            native loops, loop_num_runtime.csv, run_manifest.json
  dice/
    Raw/                 raw_dice_summary.csv, matched_loop_pairs.tsv
    iEnhance/            ienhance_dice_summary.csv, matched_loop_pairs.tsv
  apa/
    Raw/                 APA matrix, summary, heatmaps, run_manifest.json
    iEnhance/            APA matrix, summary, heatmaps, run_manifest.json
  rna/
    raw/                 selected loop-gene pairs and correlation outputs
    matched/             rna_comparison.csv and paired heatmaps
  mcg/
    raw/                 selected loops and correlation outputs
    matched/             mcg_comparison.csv and paired heatmaps
  minmax/                Fig6d_minmax, SupFig7_minmax and component score tables
```

Existing inputs and completed results are supplied through the configuration.
This edition does not bundle the experimental data or result directories. To collect
the saved two-workflow example from a terminal, the separate existing entry is:

```bash
python call_loop/tool/scripts/run_loop_workflow_examples.py \
  --config "$LOOP_WORKFLOW_CONFIG"
```

The collected metric table contains:

Source: `F:/project/test1/final_code2/call_loop/result/loop_workflow_examples/workflow_summary.csv` (first 3 lines).

```text
loop_method,impute_method,loop_num,runtime_seconds,mapped_reference_Dice,APA_P2LL,RNA_Pearson_r,mCG_Pearson_r
Mustache,Raw,40,410.11916300002486,0.0045103857566765,7.147050014974543,0.2508746438490196,-0.2135770499902065
Mustache,iEnhance,24595,1082.4974068,0.0461491813220133,4.880963830726539,0.259202430683774,-0.2610454627419818
```


## 14. Implementation and tutorial references

- Caller command templates ([call_loop/tool/callers/callers.py](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/tool/callers/callers.py) in this folder)
- Metric implementations and Fig. 6c workflow ([call_loop/tool/get_result.py](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/tool/get_result.py) in this folder)
- Loop metric guide ([Loop tutorial](loop.ipynb))
- Input formats and existing local data ([call_loop/DATA_REQUIREMENTS.md](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/DATA_REQUIREMENTS.md) in this folder)
- [software notes](#Software-and-implementation)

The presentation follows the stepwise organization of the scHiCluster
[imputation tutorial](https://zhoujt1994.github.io/scHiCluster/Tan2021/imputation.html)
and [loop-matrix analysis tutorial](https://zhoujt1994.github.io/scHiCluster/hba/loop_majortype/01.diff.html).
The analysis commands, metric definitions and outputs described here come from
this repository's Loop workflow.


## Software and implementation

| Stage | Software context |
|---|---|
| Dice, APA and prepared-input RNA | Python evaluation runtime and the source helpers |
| Mustache native example | Isolated Mustache runtime |
| Other native callers | Each caller's documented environment and checkpoint |
| mCG extraction | Ubuntu/WSL ALLCools environment |

The complete runtime inventory remains at [call_loop/ENVIRONMENT.md](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/ENVIRONMENT.md) in
the this folder. The examples above show the actual interfaces of
the caller and metric helpers; this reading edition does not import them.

Return to the [tutorial index](tutorials.ipynb).

[Environment instructions](https://github.com/rawandy2001-design/benchmark/blob/main/call_loop/ENVIRONMENT.md) · [Repository guide](../README.md)
